# Week 3: Intelligence Layer — Win Alignment, Benchmarks & Optimization

This notebook demonstrates the full Week 3 intelligence pipeline:

1. Start with a **weak pipeline** — low distress, few states, few sectors
2. **Benchmark** it against historical NMTC winner patterns
3. **Score alignment** with the `WinProbabilityModel`
4. **Get quantified recommendations** to improve competitiveness
5. **Optimize** the pipeline subset to maximize winner alignment
6. **Compare** before vs. after

> **IMPORTANT — Data Limitation:** Scores in this notebook reflect *alignment with historical winner patterns* (CY2020–2024), **not win probability**. The CDFI Fund does not publish non-winner application data, so a true win probability cannot be computed. See `methodology_disclosure` on any score object.

In [1]:
import sys
sys.path.insert(0, '..')

from nmtcapp.core.application import Application
from nmtcapp.core.cde import CDEProfile
from nmtcapp.core.pipeline import Pipeline, PipelineProject

## 1. Build a Weak Pipeline

Start with a pipeline that has common problems: concentrated in one state, one sector, and mostly standard-LIC distress.

In [2]:
cde = CDEProfile.sample()

# Build a weak pipeline: single state, single sector, LIC-only distress.
# Eligibility is pre-set so no external API calls are triggered.
weak_pipeline = Pipeline()
for i in range(8):
    p = PipelineProject(
        project_id=f"WEAK-{i:03d}",
        project_name=f"Illinois Community Facility Project {i+1}",
        qalicb_name=f"IL QALICB {i+1}",
        address=f"{100+i} S Michigan Ave",
        city="Chicago",
        state="IL",                    # single state — geographic concentration
        sector="community_facility",   # single sector — no diversity
        project_type="real_estate",
        total_project_cost=8_000_000,
        qei_request=5_000_000,
        qlici_amount=5_000_000,
        expected_jobs_created=8,       # low impact intensity (1.6 jobs/$MM)
        # Pre-set eligibility — LIC only, no deep/severe distress
        is_nmtc_eligible=True,
        distress_level="lic",
        is_native_area=False,
        is_high_migration_rural=False,
        is_opportunity_zone=False,
    )
    weak_pipeline.add(p)

app = Application(cde=cde, requested_allocation=45_000_000)
app.add_pipeline(weak_pipeline)
print(f"Weak pipeline: {len(weak_pipeline)} projects, ${sum(p.qei_request for p in weak_pipeline):,.0f} total QEI")
print(f"  States: {len({p.state for p in weak_pipeline})}  |  "
      f"Sectors: {len({p.sector for p in weak_pipeline})}  |  Distress: LIC-only")


Weak pipeline: 8 projects, $40,000,000 total QEI
  States: 1  |  Sectors: 1  |  Distress: LIC-only


## 2. Benchmark Against Historical Winners

In [3]:
bc = app.benchmark()
print(bc.summary())

  BENCHMARK vs. HISTORICAL NMTC WINNERS (CY2020–CY2024)
  Overall Alignment Score: 11.2 / 100

  [+] STRONG             1 metric(s)
  [~] COMPETITIVE        1 metric(s)
  [-] WEAK               2 metric(s)
  [!] BELOW_WEAK         5 metric(s)

  Metric                                Value   Tier          Pctile
  ---------------------------------- --------   ------------  ------
  [!] Deep/Severe Distress %               0.00   below_weak    0%
  [!] States Served                        1.00   below_weak    5%
  [!] Geographic HHI                   10000.00   below_weak    0%
  [!] Jobs per $1MM QEI                    1.60   below_weak    4%
  [!] Max Single Sector %                  1.00   below_weak    0%
  [-] Sectors Represented                  1.00   weak          0%
  [~] Pipeline Projects                    8.00   competitive   5%
  [+] NMTC Eligibility %                   1.00   strong        84%
  [-] Rural % of QEI                       0.00   weak          4%

  Methodology

In [4]:
# Inspect the tier breakdown
print("Tier summary:", bc.tier_summary)
print(f"\nOverall alignment score: {bc.overall_benchmark_score:.1f}/100")
print(f"\nMethodology: {bc.methodology_disclosure[:200]}...")

Tier summary: {'strong': 1, 'competitive': 1, 'weak': 2, 'below_weak': 5}

Overall alignment score: 11.2/100

Methodology: Benchmarks compare input metrics against patterns observed in CDFI Fund NMTC award announcements (CY2020–CY2024). Only winner-level data is publicly available; non-winner distributions are unknown. Sc...


## 3. Score Win Alignment

The `WinProbabilityModel` computes a 5-dimension alignment score. **This is NOT a win probability** — it measures how closely the application resembles historical winners.

In [5]:
score = app.score_win_probability()
print(score.summary())

  APPLICATION ALIGNMENT SCORE (vs. Historical NMTC Winners)
  Composite Score:    5.5 / 100
  Competitive Tier:   WEAK
  Acceptance Baseline:33.6% (recent 4-round average)

  Dimensional Scores:
  Distress Concentration           0.0  ░░░░░░░░░░░░
  Geographic Diversity             2.6  ░░░░░░░░░░░░
  Impact Intensity                 4.2  ░░░░░░░░░░░░
  Sector Diversity                 0.0  ░░░░░░░░░░░░
  Pipeline Quality                77.7  █████████░░░

  Assessment: Below typical winner patterns — significant changes required. Standout dimensions: pipeline quality. Priority gaps: distress concentration, geographic diversity.

  *** METHODOLOGY NOTE ***
  IMPORTANT: This score measures alignment with patterns observed in historical NMTC award winners (CY2020–CY2024), not probability of selection. The CDFI Fund does not publish non-winner application data, so a true win probability cannot be computed. A high alignment score improves competitiveness but does not guarantee an award. Al

In [6]:
# Key numbers
print(f"Composite alignment score: {score.composite_score:.1f}/100")
print(f"Competitive tier:          {score.competitive_tier.upper()}")
print(f"Historical acceptance rate:{score.acceptance_rate_baseline:.1%} (recent 4-round avg)")
print("")
print("Dimensional scores:")
for dim, val in score.dimensional_scores.items():
    bar = '█' * int(val / 10) + '░' * (10 - int(val / 10))
    print(f"  {dim.replace('_', ' ').title():<30} {val:5.1f}  [{bar}]")

Composite alignment score: 5.5/100
Competitive tier:          WEAK
Historical acceptance rate:33.6% (recent 4-round avg)

Dimensional scores:
  Distress Concentration           0.0  [░░░░░░░░░░]
  Geographic Diversity             2.6  [░░░░░░░░░░]
  Impact Intensity                 4.2  [░░░░░░░░░░]
  Sector Diversity                 0.0  [░░░░░░░░░░]
  Pipeline Quality                77.7  [███████░░░]


## 4. Get Quantified Recommendations

In [7]:
recs = app.recommendations()
print(recs.summary())

  RECOMMENDATIONS
  Weak alignment (6/100). Pipeline restructuring required to be competitive. All 'critical' recommendations are blocking issues.

  [!] CRITICAL PRIORITY
  ──────────────────────────────────────────────────────────────
  Category:  Distress
  Finding:   Deep/severe distress concentration is 0% — below the p25 floor of 72% seen in historical winners.
  Action:    Replace at least 72 percentage points of standard-LIC pipeline with projects in Deep Distressed or Severely Distressed census tracts. Use CDFI Fund's NMTC Mapping Tool to identify qualifying tracts.
  Impact:    Bring distress concentration into the competitive range; NOFA scoring criteria weight this dimension heavily.
  Estimate:  Estimated +15–25 distress alignment score points; moves from 'below_weak' tier toward 'competitive'.

  Category:  Geographic
  Finding:   Pipeline spans only 1 state(s) — below the minimum of 2 states seen in any historical winner.
  Action:    Immediately add projects in at least

In [8]:
# Quick access to the critical items
critical = [r for r in recs.recommendations if r.priority == 'critical']
print(f"Critical recommendations: {len(critical)}")
for r in critical:
    print(f"  [{r.category.upper()}] {r.finding[:80]}")
    print(f"    → {r.action[:100]}")
    print(f"    Estimate: {r.quantified_improvement}")
    print()

Critical recommendations: 2
  [DISTRESS] Deep/severe distress concentration is 0% — below the p25 floor of 72% seen in hi
    → Replace at least 72 percentage points of standard-LIC pipeline with projects in Deep Distressed or S
    Estimate: Estimated +15–25 distress alignment score points; moves from 'below_weak' tier toward 'competitive'.

  [GEOGRAPHIC] Pipeline spans only 1 state(s) — below the minimum of 2 states seen in any histo
    → Immediately add projects in at least 2 additional state(s). A single-state pipeline is rarely funded
    Estimate: Geographic score near 0 currently; +30–50 points by reaching 3 states.



## 5. Pattern Analysis — What Do Winners Look Like?

Before optimizing, let's understand the winner benchmark data.

In [9]:
from nmtcapp.intelligence.pattern_analysis import analyze_winning_patterns, compare_to_winners
from nmtcapp.data.historical_awards import get_historical_winners

# Historical round data
df = get_historical_winners()
print("CDFI Fund NMTC Allocation Rounds (CY2020–2024):")
print(df[['round', 'applications', 'awards', 'acceptance_rate', 'avg_award']].to_string(index=False))

CDFI Fund NMTC Allocation Rounds (CY2020–2024):
 round  applications  awards  acceptance_rate  avg_award
CY2020           196      76            0.388   65789000
CY2021           341     100            0.293   50000000
CY2022           280     100            0.357   50000000
CY2023           305     107            0.351   48131000
CY2024           320     110            0.344   45455000


In [10]:
patterns = analyze_winning_patterns()
print("Historical winner distress patterns:")
d = patterns['distress']
print(f"  Deep/severe: p25={d['p25_pct_deep_or_severe']:.0%}  p50={d['p50_pct_deep_or_severe']:.0%}  p75={d['p75_pct_deep_or_severe']:.0%}")
print(f"\nHistorical winner geographic patterns:")
g = patterns['geographic']
print(f"  States:  mean={g['mean_states']:.1f}  p50={g['p50_states']:.0f}  p75={g['p75_states']:.0f}")
print(f"  HHI:     mean={g['mean_hhi']:.0f}")
print(f"\nHistorical winner impact benchmarks:")
i = patterns['impact']
print(f"  Jobs/$MM: p25={i['p25_jobs_per_mm_qei']:.0f}  p50={i['p50_jobs_per_mm_qei']:.0f}  p75={i['p75_jobs_per_mm_qei']:.0f}")

Historical winner distress patterns:
  Deep/severe: p25=72%  p50=82%  p75=91%

Historical winner geographic patterns:
  States:  mean=7.2  p50=7  p75=10
  HHI:     mean=620

Historical winner impact benchmarks:
  Jobs/$MM: p25=6  p50=10  p75=18


In [11]:
# Where does our pipeline stand vs. winners?
analysis = app.analyze()
comparison = compare_to_winners(analysis.pipeline_result)

print("Gap to winner medians:")
print(f"  Distress:  observed={comparison['distress']['observed_pct_deep_or_severe']:.0%}  "
      f"winner_p50={comparison['distress']['winner_p50']:.0%}  "
      f"label={comparison['distress']['gap_label']}")
print(f"  States:    observed={comparison['geographic']['observed_states']}  "
      f"winner_p50={comparison['geographic']['winner_p50_states']}  "
      f"gap={comparison['geographic']['gap_to_winner_median_states']}")
print(f"  Jobs/$MM:  observed={comparison['impact']['observed_jobs_per_mm_qei']:.1f}  "
      f"winner_p50={comparison['impact']['winner_p50']:.0f}  "
      f"label={comparison['impact']['gap_label']}")

Gap to winner medians:
  Distress:  observed=0%  winner_p50=82%  label=below_winner_p25
  States:    observed=1  winner_p50=7.0  gap=6
  Jobs/$MM:  observed=1.6  winner_p50=10  label=below_winner_p25


## 6. Build an Improved Pipeline

Based on the recommendations, build a stronger pipeline:

In [12]:
# Improved pipeline following the recommendations:
#   ✓ 15 states (was 1)                  → geographic diversity
#   ✓ 7 sectors (was 1)                   → sector diversity
#   ✓ 87% deep/severe distress (was 0%)   → distress concentration
#   ✓ 20 projects across diverse markets  → richer optimizer candidate pool
#
# Pipeline.sample() ships pre-validated eligibility data — no API calls needed.
improved_pipeline = Pipeline.sample(n=20)

app2 = Application(cde=cde, requested_allocation=55_000_000)
app2.add_pipeline(improved_pipeline)

states_count = len({p.state for p in improved_pipeline})
sectors_count = len({p.sector for p in improved_pipeline})
deep_qei = sum(p.qei_request for p in improved_pipeline if p.distress_level in ('deep', 'severe'))
total_qei = sum(p.qei_request for p in improved_pipeline)
print(f"Improved pipeline: {len(improved_pipeline)} projects across {states_count} states")
print(f"  Sectors:         {sectors_count}")
print(f"  Deep/Severe:     {deep_qei/total_qei:.0%} of QEI (was 0%)")
print(f"  Total QEI:       ${total_qei:,.0f}")


Improved pipeline: 20 projects across 20 states
  Sectors:         7
  Deep/Severe:     87% of QEI (was 0%)
  Total QEI:       $122,500,000


In [13]:
# Score the improved pipeline
score2 = app2.score_win_probability()
print(f"BEFORE: {score.composite_score:.1f}/100 [{score.competitive_tier}]")
print(f"AFTER:  {score2.composite_score:.1f}/100 [{score2.competitive_tier}]")
print(f"DELTA:  {score2.composite_score - score.composite_score:+.1f} points")
print()
print("Dimensional comparison:")
for dim in score.dimensional_scores:
    b = score.dimensional_scores[dim]
    a = score2.dimensional_scores[dim]
    print(f"  {dim.replace('_', ' ').title():<30} {b:5.1f} → {a:5.1f}  ({a-b:+.1f})")

BEFORE: 5.5/100 [weak]
AFTER:  65.9/100 [competitive]
DELTA:  +60.4 points

Dimensional comparison:
  Distress Concentration           0.0 →  69.8  (+69.8)
  Geographic Diversity             2.6 →  83.0  (+80.4)
  Impact Intensity                 4.2 →  20.5  (+16.3)
  Sector Diversity                 0.0 → 100.0  (+100.0)
  Pipeline Quality                77.7 →  94.5  (+16.8)


## 7. Pipeline Optimizer

The optimizer selects a subset of projects from the pipeline to maximize alignment with historical winners, subject to QEI budget and diversity constraints.

In [14]:
from nmtcapp.optimizer import OptimizationConstraints, PipelineOptimizer

# Use the improved pipeline and optimize to $45M budget with at least 6 states
constraints = OptimizationConstraints(
    min_total_qei=35_000_000,
    max_total_qei=50_000_000,
    min_projects=8,
    min_states=6,
    required_sectors=["healthcare"],
)

result = PipelineOptimizer(max_iterations=200).optimize(
    improved_pipeline, constraints, 45_000_000
)
print(result.summary())

  PIPELINE OPTIMIZATION RESULT
  Projects selected:    9
  Total QEI:            $49,500,000
  Alignment (before):   76.0 / 100
  Alignment (after):    78.7 / 100
  Improvement:          +2.6 pts
  Constraints met:      Yes
  Search iterations:    10

  Dimensional Improvements:
    Distress                       +7.3 pts
    Geographic                     -18.6 pts
    Impact                         +18.6 pts
    Sector                         -2.7 pts
    Pipeline                       -9.2 pts

  NOTE: Pipeline optimized using greedy construction (projects ranked by individual alignment contribution) followed by swap-bas...


In [15]:
# Which projects were selected?
print("Selected projects:")
for p in result.selected_projects:
    print(f"  {p.project_id}  {p.state}  {p.sector:<22}  ${p.qei_request:,.0f}  {p.expected_jobs_created} jobs")

total_qei = sum(p.qei_request for p in result.selected_projects)
total_jobs = sum(p.expected_jobs_created for p in result.selected_projects)
jpm = total_jobs / (total_qei / 1_000_000)
states_selected = len({p.state for p in result.selected_projects})
print(f"\nTotal QEI: ${total_qei:,.0f}  |  Jobs/$MM: {jpm:.1f}  |  States: {states_selected}")

Selected projects:
  PRJ-019  MD  community_facility      $4,200,000  45 jobs
  PRJ-010  TN  small_business          $3,500,000  90 jobs
  PRJ-004  CA  small_business          $4,500,000  80 jobs
  PRJ-002  TX  education               $7,000,000  38 jobs
  PRJ-003  NY  small_business          $5,000,000  65 jobs
  PRJ-015  MI  small_business          $5,000,000  110 jobs
  PRJ-008  PA  healthcare              $8,000,000  55 jobs
  PRJ-020  NM  mixed_use               $5,800,000  32 jobs
  PRJ-014  AZ  healthcare              $6,500,000  48 jobs

Total QEI: $49,500,000  |  Jobs/$MM: 11.4  |  States: 9


## 8. Application via the Full `Application.optimize_pipeline()` Interface

In [16]:
# Same optimizer, accessed directly from the Application object
opt_result = app2.optimize_pipeline(
    constraints=OptimizationConstraints(
        max_total_qei=45_000_000,
        min_states=5,
    )
)

print(f"Alignment before: {opt_result.alignment_score_before * 100:.1f}/100")
print(f"Alignment after:  {opt_result.alignment_score_after * 100:.1f}/100")
print(f"Improvement:      {(opt_result.alignment_score_after - opt_result.alignment_score_before) * 100:+.1f} pts")
print(f"Feasible:         {opt_result.constraints_satisfied}")
print(f"Iterations:       {opt_result.iterations}")
print()
print("Dimensional improvements:")
for dim, delta in opt_result.dimensional_improvements.items():
    arrow = '↑' if delta > 0 else ('↓' if delta < 0 else '—')
    print(f"  {dim.replace('_', ' ').title():<30} {arrow} {delta * 100:+.1f}")

Alignment before: 76.0/100
Alignment after:  79.1/100
Improvement:      +3.1 pts
Feasible:         True
Iterations:       9

Dimensional improvements:
  Distress                       ↑ +7.3
  Geographic                     ↓ -18.4
  Impact                         ↑ +18.6
  Sector                         — +0.0
  Pipeline                       ↓ -9.2


## 9. Methodology Disclosure

Always check the methodology disclosure before interpreting any score:

In [17]:
print("=" * 70)
print("METHODOLOGY DISCLOSURE")
print("=" * 70)
print(score2.methodology_disclosure)
print()
print("Benchmark methodology:")
print(bc.methodology_disclosure)

METHODOLOGY DISCLOSURE
IMPORTANT: This score measures alignment with patterns observed in historical NMTC award winners (CY2020–CY2024), not probability of selection. The CDFI Fund does not publish non-winner application data, so a true win probability cannot be computed. A high alignment score improves competitiveness but does not guarantee an award. Alignment score ≠ win probability.

Benchmark methodology:
Benchmarks compare input metrics against patterns observed in CDFI Fund NMTC award announcements (CY2020–CY2024). Only winner-level data is publicly available; non-winner distributions are unknown. Scores reflect alignment with historical winners, not probability of selection. Use as diagnostic guidance only — not as a prediction of funding outcomes.


## Summary

This notebook demonstrated the full Week 3 intelligence workflow:

| Step | Tool | What You Get |
|---|---|---|
| Benchmark | `app.benchmark()` | 9-metric tier comparison vs. CY2020-2024 winners |
| Alignment Score | `app.score_win_probability()` | 0-100 score across 5 dimensions |
| Recommendations | `app.recommendations()` | Quantified, prioritized improvement actions |
| Pattern Analysis | `compare_to_winners()` | Gap-to-winner-median per dimension |
| Optimization | `app.optimize_pipeline()` | Max-alignment project subset |

**Key reminder:** All scores reflect *alignment with historical NMTC winner patterns*. No true win probability can be computed from public data alone.